# Assignment 2 — Stage 1: Sentiment Classification of Movie Reviews

**CYSE499/650, Summer 2026**

The task is binary sentiment classification over the Pang & Lee movie-review polarity
corpus (`0 = negative`, `1 = positive`). The released training split is deliberately
small and skewed, which is the real difficulty of the assignment:

| Split | Size | Positive | Negative |
|---|---|---|---|
| `train.csv` | 240 | 180 | 60 |
| `public_test.csv` | 400 | 200 | 200 |

This notebook documents the model, the design decisions forced by that data, the
cross-validated comparison used to choose between candidates, and the final evaluation
on the public test set.

**Reproducibility note.** Training is performed by `train.py`, which writes
`model_checkpoint/` and `results.json`. This notebook *loads* those artefacts rather
than retraining, so it runs in seconds and reports exactly the model that was submitted.
Re-run `python train.py` to regenerate them from scratch.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
train = pd.read_csv(ROOT / "data" / "train.csv")
public_test = pd.read_csv(ROOT / "data" / "public_test.csv")

print("train      ", train.shape, dict(train.label.value_counts().sort_index()))
print("public_test", public_test.shape, dict(public_test.label.value_counts().sort_index()))

## 1. What the data actually looks like

Two properties of this corpus drove every design decision below, so they are measured
here rather than assumed.

In [ ]:
words = train.text.str.split().str.len()
print("review length in words — median %d, mean %d, 90th pct %d, max %d"
      % (words.median(), words.mean(), words.quantile(0.9), words.max()))

# Sanity check: no document is shared between the two splits.
overlap = set(train.source_file) & set(public_test.source_file)
print("documents shared between train and public test:", len(overlap))

print("\nfirst 300 characters of one review:")
print(repr(train.text.iloc[0][:300]))

Two consequences:

**(a) The reviews are long.** A median review is ~730 words (~1,000 word-piece tokens)
and the longest is 2,570 words. A pretrained encoder with a 512-token window would
discard the majority of most reviews — and in this corpus the verdict often arrives in
the final paragraph, after several paragraphs of plot summary. Any fixed-window model
would need chunking plus document-level aggregation to avoid throwing away the part that
carries the label.

**(b) There is very little supervision.** 240 labelled documents is small enough that a
high-capacity model memorises the training set almost immediately.

Together these pushed the design toward representations that read the **whole** document
and models with **few effective parameters**.

> **Label leakage warning.** The `id` column encodes the gold label
> (`pos_cv696_29740`, `neg_cv963_7208`). It is used only to format the output CSV and is
> never given to a model. Likewise `label_name` and `source_file` are never used as
> features.

## 2. Model structure

Two candidate models were built over a shared text representation, and a weighted blend
of them was also evaluated.

**Shared representation — TF-IDF over the full review**

- word 1–2 grams, `min_df=2`, sublinear term frequency
- character `char_wb` 3–5 grams, `min_df=3`, sublinear term frequency

Character n-grams matter here specifically because of the small training set: with only
240 documents the learned word vocabulary is thin, and the assignment warns that
evaluation data contains tokens unseen in training. Character n-grams degrade gracefully
on unseen words (they still match morphology and sub-word fragments) where a pure word
model simply drops them.

**Model A — Logistic regression** directly on the sparse TF-IDF vector, with
`class_weight="balanced"`.

**Model B — Feed-forward neural network (PyTorch).** The sparse TF-IDF matrix is reduced
with truncated SVD (latent semantic analysis) and L2-normalised, then fed to:

```
Dropout(0.5) → Linear(d → 256) → LayerNorm → GELU → Dropout(0.5) → Linear(256 → 2)
```

`svd_components` is requested as 300, but truncated SVD cannot produce more components
than there are documents, so the effective width is **d = 240** — a detail worth stating
because it means the representation is saturated: every additional latent dimension the
configuration asks for is unavailable at this dataset size.

The SVD step is what makes a neural model viable on 240 examples: it collapses ~57k
sparse features into a dense subspace, so the network has a tractable number of input
weights and the latent dimensions capture co-occurrence structure that individual
n-grams cannot.

**Model C — Blend.** A weighted average of the two probability outputs, with the weight
chosen on cross-validation.

In [ ]:
import inspect
import predict
import train as training

print(inspect.getsource(training.make_tfidf))
print(inspect.getsource(predict.build_mlp))

## 3. How the small and imbalanced training set was handled

The training split is 75% positive while both evaluation sets are 50/50. Left alone, a
model trained on this prior scores well on training-like data by leaning positive, and
then loses most of that advantage on a balanced test set. Four separate mechanisms
address this.

**1. Class-weighted loss.** Logistic regression uses `class_weight="balanced"`; the
neural model uses `CrossEntropyLoss(weight=...)` with the same inverse-frequency
weights. Each negative example contributes 3× the gradient of a positive one, so the
model cannot reach a low loss by predicting "positive" everywhere.

**2. Decision threshold tuned for balanced accuracy.** The probability cutoff is *not*
left at 0.5. It is chosen to maximise balanced accuracy on **out-of-fold** predictions,
then frozen into the checkpoint and applied unchanged at inference. This is the single
largest correction for the prior mismatch, and the notebook reports the model at both
0.5 and the tuned cutoff so the effect is visible.

**3. Repeated stratified cross-validation for every decision.** With 240 examples a
single train/validation split is mostly noise. Model choice, the regularisation
strength, the blend weight, and the threshold are all selected on repeated stratified
5-fold CV, with each document's out-of-fold probabilities averaged across repeats.
Every fold preserves the 3:1 class ratio.

**4. Capacity kept deliberately low.** Strong L2 regularisation on the linear model;
SVD compression, dropout of 0.5 on both the input and hidden layer, weight decay, and
gradient clipping on the neural model. The aim is a model that cannot memorise 240
documents.

**Selection metric.** Balanced accuracy, not raw accuracy — on a 3:1 training prior,
raw accuracy rewards exactly the bias we are trying to remove.

**What `public_test.csv` was used for: nothing but the final report.** It is not used to
fit, tune, threshold, blend, or early-stop. The rules forbid training on it, and every
number in Section 6 comes from a model that had never seen it.

## 4. Key training techniques

Hyper-parameters for the neural model, taken directly from `train.py`:

| Setting | Value | Why |
|---|---|---|
| Optimizer | AdamW | Decoupled weight decay; the standard choice for small dense classifiers |
| Learning rate | `1e-3` | With only 15 optimizer steps per epoch, a smaller LR does not converge inside the epoch budget |
| LR schedule | Cosine annealing to 0 over 30 epochs | Removes "which epoch to stop at" as a tunable, which matters when there is no validation set to spare |
| Weight decay | `1e-4` | Regularisation on top of dropout |
| Batch size | 16 | 240 examples → 15 steps/epoch; larger batches give too few updates, smaller ones are unstable |
| Epochs | 30 | Fixed budget, paired with the cosine schedule |
| Loss | Cross-entropy, inverse-frequency class weights | Counters the 3:1 skew |
| Gradient clipping | max-norm 1.0 | Guards against outlier batches in a small dataset |
| Dropout | 0.5 (input and hidden) | The main capacity control |
| Seed | 20260804 | Fixed for reproducibility |

For logistic regression the tuned parameter is the inverse regularisation strength `C`,
selected over `{3, 10}` by cross-validation with `liblinear`.

In [ ]:
print(inspect.getsource(training.train_mlp))

## 5. Cross-validated model comparison

All numbers below come from `train.py` and use `train.csv` only.

In [ ]:
results = json.load(open(ROOT / "results.json"))
print("CV protocol:", results["cv_protocol"])

cv = pd.DataFrame(results["cv_table"])[
    ["model", "roc_auc", "bal_acc@0.5", "bal_acc@tuned", "macro_f1@tuned", "threshold"]
]
cv

In [ ]:
sel = results["selected"]
print("Selected model :", sel["detail"]["model"])
print("Blend weights  :", sel["weights"])
print("CV balanced acc: %.4f" % sel["detail"]["bal_acc@tuned"])
print("Threshold      : %.4f" % sel["detail"]["threshold"])

Note the gap between the `bal_acc@0.5` and `bal_acc@tuned` columns — that difference is
the imbalance correction described in Section 3, measured rather than assumed.

## 6. Evaluation on the public test set

The checkpoint is reloaded from disk exactly the way a grader would load it, then scored
on all 400 public test reviews.

In [ ]:
from predict import load_model, predict as predict_labels, predict_proba

bundle = load_model(ROOT / "model_checkpoint")
print("checkpoint config:")
print(json.dumps(bundle.config, indent=2))

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, roc_auc_score)

texts = public_test.text.astype(str).tolist()
y_true = public_test.label.to_numpy()

y_prob = predict_proba(bundle, texts)
y_pred = predict_labels(bundle, texts)

print("Total accuracy    : %.4f" % accuracy_score(y_true, y_pred))
print("Balanced accuracy : %.4f" % balanced_accuracy_score(y_true, y_pred))
print("ROC AUC           : %.4f" % roc_auc_score(y_true, y_prob))
print()
print(classification_report(y_true, y_pred, target_names=["negative (0)", "positive (1)"], digits=4))

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm, index=["true negative", "true positive"],
                   columns=["pred negative", "pred positive"]))

fig, ax = plt.subplots(figsize=(4.2, 3.8))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=15)
ax.set_xticks([0, 1], ["negative", "positive"])
ax.set_yticks([0, 1], ["negative", "positive"])
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Public test confusion matrix")
plt.tight_layout()
plt.show()

### Reading the confusion matrix

The two off-diagonal cells are the two failure modes. Because the training prior is 3:1
positive, the risk this model was built to avoid is an excess of false *positives* —
negative reviews predicted positive. Comparing the two off-diagonal counts shows how
much of that bias the class weighting and tuned threshold actually removed.

## 7. Writing `public_test_predictions.csv`

The submitted file is regenerated here through the same `predict.py` entry point the
grader would call, and checked against the required schema.

In [ ]:
from predict import write_predictions

out = write_predictions(ROOT / "data" / "public_test.csv",
                        ROOT / "public_test_predictions.csv", bundle=bundle)

assert list(out.columns) == ["id", "predicted_label"], out.columns
assert len(out) == len(public_test) == 400
assert set(out.predicted_label.unique()) <= {0, 1}
assert out.id.tolist() == public_test.id.tolist()
print("public_test_predictions.csv OK —", len(out), "rows")
out.head()

## 8. Limitations and honest scope

- **No pretrained transformer was used.** The plan was to compare against a fine-tuned
  MiniLM encoder with chunked document aggregation, but `transformers` could not be
  installed in this environment, so that arm was dropped rather than reported
  untested. The neural model here is trained from scratch on 240 documents; a pretrained
  language model is the most obvious source of further gains and is the first thing
  Stage 2 lists as future work.
- **The hyper-parameter sweep is small** (two values of `C`, one neural configuration),
  bounded by CPU-only training time. The neural arm used 5-fold CV without repeats,
  so its CV estimate is noisier than the logistic regression estimate.
- **240 training documents** means every CV estimate carries a standard error of roughly
  ±3 points. Differences smaller than that between the candidate models should not be
  treated as meaningful.

## References

- B. Pang and L. Lee, *A Sentimental Education: Sentiment Analysis Using Subjectivity
  Summarization Based on Minimum Cuts*, ACL 2004 — the source corpus.
- scikit-learn documentation: `TfidfVectorizer`, `TruncatedSVD`, `LogisticRegression`,
  `RepeatedStratifiedKFold`.
- PyTorch documentation: `AdamW`, `CrossEntropyLoss` class weighting,
  `CosineAnnealingLR`.

## Use of AI

Anthropic's Claude (via Claude Code) was used as a coding assistant to draft the code and
explanatory prose in this repository. All reported numbers were produced by executing that
code against the released data, and the modelling decisions were reviewed and accepted by
the author, who is responsible for this submission.